> **Denya Sree Adapa (25111595) - TESTING, MERGING & HTML EXPORT**: this notebook has only my section. In the final `01_ETL_Pipeline.ipynb`, it is pasted in order after the earlier sections, and everything runs top to bottom.

# PerfectPetClub – Customer Churn ETL Pipeline

**Module:** Data Engineering Team Presentation &nbsp;|&nbsp; **Notebook 1 of 2:** ETL pipeline

**Goal:** build a clean, one-row-per-customer dataset for analysing **customer churn**, where churn means a customer has **cancelled the auto-renew** of their subscription (`subscription.auto_renew == 0`).

| Section | Owner | What it does |
|---|---|---|
| 0. Setup & configuration | Team | Imports, paths, logging |
| 1. Extract | Bhavana Padala | Read SQLite DB, CRM CSV, server log CSV, survey CSV + question metadata |
| 2. Transform A – data quality & cleaning | Siddu Gundeboina | Data quality checks, type conversion, parsing, cleaning |
| 3. Transform B – feature engineering | Pravalika | Build customer-level features and the churn target |
| 4. Load | Pravalika | Write and verify the Parquet files |
| 5. Testing & validation | Denya Sree Adapa | Unit tests and end-to-end checks on the output |
| Repo, README, merge, HTML export | Ramya Sree Kasula & Denya Sree Adapa | Git repository, merging branches, run-all + HTML export |

**Architecture**

```
 SQLite DB ─┐
 CRM CSV  ──┤   extract_*()  ──►  raw dict  ──►  run_quality_checks() ──► clean_*()  ──►  build_*_features()
 Server log ┤                                   (DQ report)               (typed,         (one row per customer)
 Survey CSV ┘                                                              parsed)               │
                                                                                                  ▼
                                                               load_parquet()  ──►  output/customer_churn.parquet
                                                                                    output/customer_churn_model_ready.parquet
```

## 0. Setup & configuration

In [ ]:
import ast
import logging
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# ---- Configuration (change paths here, not inside functions) ----
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

SOURCES = {
    "db": DATA_DIR / "PerfectPetClub.db",
    "crm": DATA_DIR / "PerfectPetClub_CRM_report_1.csv",
    "server_log": DATA_DIR / "PerfectPetClub_server_log_1.csv",
    "survey": DATA_DIR / "PerfectPetClub_survey.csv",
    "survey_questions": DATA_DIR / "PerfectPetClub_survey_questions.txt",
}
DB_TABLES = ["customer", "subscription", "subscription_type", "sale", "sale_product",
             "product", "product_category", "payment", "product_review"]

OUTPUT_FILE = OUTPUT_DIR / "customer_churn.parquet"
MODEL_READY_FILE = OUTPUT_DIR / "customer_churn_model_ready.parquet"

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
log = logging.getLogger("ppc_etl")
log.info("pandas %s | numpy %s", pd.__version__, np.__version__)

## 5. Testing & validation  *(owner: Denya Sree Adapa)*

Three layers of testing:
1. **Unit tests** check each parser on normal, edge and invalid inputs.
2. **Error-handling tests** confirm the pipeline fails clearly when inputs are wrong.
3. **Output tests** confirm the final Parquet meets the dataset contract (grain, target, no nulls in key features, sensible ranges).

In [ ]:
results = []

def assert_(condition, message="assertion failed"):
    """assert as a function, so it can be used inside a lambda."""
    if not condition:
        raise AssertionError(message)

def test(name, fn):
    """Tiny test runner: records PASS/FAIL instead of stopping at the first failure."""
    try:
        fn()
        results.append((name, "PASS", ""))
    except Exception as e:
        results.append((name, "FAIL", repr(e)))

def _raises(exc_type, fn):
    try:
        fn()
    except exc_type:
        return
    raise AssertionError(f"{exc_type.__name__} not raised")

# ---- 1. Unit tests ----
test("parse_income normal band", lambda: assert_(parse_income("26k-30k") == 28_000))
test("parse_income open-ended top band", lambda: assert_(parse_income("198k-200k+") == 199_000))
test("parse_income invalid -> NaN", lambda: assert_(np.isnan(parse_income("prefer not to say"))))
test("parse_platform list-string", lambda: assert_(parse_platform("['live-chat']") == "live-chat"))
test("parse_platform malformed", lambda: assert_(parse_platform("['e-mail'") == "e-mail"))
test("parse_user_agent", lambda: assert_(parse_user_agent('"Mozilla/5.0 (Tablet; Zenith ZenithPad) AppleWebKit') == ("Tablet", "Zenith ZenithPad")))
test("parse_user_agent unknown", lambda: assert_(parse_user_agent("curl/8.0") == ("Unknown", "Unknown")))

# ---- 2. Error handling ----
test("missing file raises FileNotFoundError", lambda: _raises(FileNotFoundError, lambda: extract_csv(Path("nope.csv"), "x")))
test("missing DB table raises ValueError", lambda: _raises(ValueError, lambda: extract_sqlite(SOURCES["db"], ["not_a_table"])))

# ---- 3. Output contract ----
out = pd.read_parquet(OUTPUT_FILE)
key_features = ["churned", "subscription_type", "tenure_days", "orders_per_month", "avg_star_rating",
                "complaints_per_month", "logins_per_month", "nps_recommend", "income_gbp"]
test("one row per customer", lambda: assert_(out["customer_id"].is_unique and len(out) == len(raw["customer"])))
test("target is binary", lambda: assert_(set(out["churned"].unique()) <= {0, 1}))
test("target matches source auto_renew", lambda: assert_((out["churned"] == 1 - out["auto_renew"]).all()))
test("no nulls in key features", lambda: assert_(out[key_features].notna().all().all()))
test("rates are non-negative", lambda: assert_((out[["orders_per_month", "logins_per_month", "complaints_per_month"]] >= 0).all().all()))
test("ratings within 1-5", lambda: assert_(out["avg_star_rating"].between(1, 5).all()))
test("no PII columns", lambda: assert_(not {"email", "name", "password_hash", "postcode"} & set(out.columns)))
test("model-ready has no nulls", lambda: assert_(pd.read_parquet(MODEL_READY_FILE).notna().all().all()))

test_report = pd.DataFrame(results, columns=["test", "result", "detail"])
print(f"{(test_report.result == 'PASS').sum()}/{len(test_report)} tests passed")
test_report

In [ ]:
# Final summary of the dataset handed over to the dashboard
summary = churn_df.groupby("churned")[["tenure_days", "orders_per_month", "avg_order_value_gbp", "late_delivery_rate",
                                       "avg_star_rating", "complaints_per_month", "logins_per_month",
                                       "nps_recommend", "online_comparison_freq"]].mean().T
summary.columns = ["Retained (0)", "Churned (1)"]
print(f"Customers: {len(churn_df)} | Churned: {churn_df.churned.sum()} | Churn rate: {churn_df.churned.mean():.1%}")
summary.round(2)